## 1주차: 파이썬 기초

### 과제 1
`temp_samples = [72, 75, 79, 88, 91, 85]` 리스트를 만들고, **85 초과 값만** 출력하세요.

### 과제 2
`is_overheat(temp)` 함수를 작성하고, `temp >= 90`이면 `True`, 아니면 `False`를 반환하게 만드세요.

### 과제 3
NumPy를 사용해 `temp_samples`의 최솟값/최댓값/평균을 계산하세요.

In [14]:
# 1
temp_samples = [72, 75, 79, 88, 91, 85]
# print([tmp for tmp in temp_samples if tmp > 85])
for temp in temp_samples:
    if temp > 85: print(temp)
# 2
def is_overheat(temp): return temp >= 90

# 3
import numpy as np
nptemp_samples = np.array(temp_samples)
print(nptemp_samples.min(), nptemp_samples.max(), nptemp_samples.mean())


88
91
72 91 81.66666666666667


### 미니 미션

아래 조건을 만족하는 코드를 작성해보세요.

- `speed = np.array([35, 42, 51, 49, 62, 71, 68, 55])`
- 속도 50 이상만 추출
- 추출된 값의 평균 계산
- 평균이 60 이상이면 `"고속 주행 구간"` 출력, 아니면 `"일반 주행 구간"` 출력

추가 도전:
- 인덱스까지 함께 출력해 어떤 시점에서 50 이상이었는지 확인

In [15]:
import numpy as np

speed = np.array([35, 42, 51, 49, 62, 71, 68, 55])
if speed[speed >= 50].mean() >= 60: print("고속 주행 구간")
else : print("일반 주행 구간")
print("위치:", np.where(speed >= 50)[0]) # np.where는 튜플로 반환하므로 [0]으로 위치 배열을 추출

# idx, = np.where(speed >= 50)
# print("위치:", idx)

고속 주행 구간
위치: [2 4 5 6 7]


## 2주차: IMU 센서

### 필터링 기법
1. **Average Filter**: 전체 데이터의 평균값으로 필터링
2. **Moving Average Filter**: 슬라이딩 윈도우를 사용한 이동 평균
3. **Exponential Moving Average Filter**: 가중 평균을 사용한 지수 이동 평균

In [16]:

class MotionSensorProcessor:
    """차량 모션 센서 데이터 처리 클래스"""

    def __init__(self):
        self.data = {}

    # 1. 평균 필터
    def average_filter(self, data: np.ndarray) -> np.ndarray: # 변수명은 ndarray임. np.array([])는 전환해주는 함수

        filtered_data = np.zeros_like(data)
        for i in range(len(data)):
            filtered_data[i] = data[:i+1].mean() #numpy 확실하니까 가능.
            # filtered_data[i] = np.mean(data[:i+1])

        return filtered_data
    
    # 2. 이동 평균 필터    
    def mov_avg_filter(self, data:np.ndarray, win_size: int =5) -> np.ndarray:
        if win_size <= 0: raise ValueError("윈도우 크기는 양수여야 합니다.")
        
        filtered_data = np.zeros_like(data, dtype=float) 
        # .mean()이 float을 뱉어도 filtered_data가 float여야 함. 
        # 만약 data가 int면 int로 저장됨
        for i in range(len(data)):
            start_idx = max(0, i + 1 - win_size)
            end_idx = i+1
            filtered_data[i] = data[start_idx:end_idx].mean()
        return filtered_data
    
    # 3. 지수 이동 평균 필터
    def exp_mov_avg_filter(self, data: np.ndarray, alpha: float = 0.3) -> np.ndarray:

        if alpha <= 0 or alpha > 1: raise ValueError("alpha 값은 0과 1 사이여야 합니다.")
        
        filtered_data = np.zeros_like(data, dtype=float)
        filtered_data[0] = data[0]
        
        for i in range(1, len(data)):
            filtered_data[i] = (1 - alpha) * filtered_data[i-1] + alpha * data[i]
        return filtered_data

In [ ]:
def pulse_counting_method(self, time:np.ndarray, pulses:np.ndarray, pulses_per_revolution: int, counting_time_interval: float) ->np.ndarray:

	est_ang_vel = np.zeros(len(time))

	tmp_pulse_count = 0
	curr_est_ang_vel = 0.0
	prev_time = time[0] # 미리 초기화
	prev_pulse =pulses[0] # 미리 초기화

	for idx in range(1, len(time)):
		if pulses[idx] == 1 and prev_pulse == 0: # 스칼라간의 비교는 'and'
			tmp_pulse_count += 1

		delta_time = time[idx] - prev_time
		
		if delta_time >= counting_time_interval:
			curr_est_ang_vel = tmp_pulse_count * 2 * np.pi / (pulses_per_revolution * counting_time_interval)
			tmp_pulse_count = 0
			prev_time = time[idx]
		 
		est_ang_vel[idx] = curr_est_ang_vel
		prev_pulse = pulses[idx]
		
	return est_ang_vel

### 3. 휠 엔코더 시뮬레이션 및 속도 추정

**Pulse Timing Method (T-Method)**: 펄스 간 시간 간격으로 속도 추정
**T-Method**: N = 1, ΔT = 펄스 간 시간 간격  
### 수식
**공통 각속도 계산:**
휠 엔코더로부터 각속도를 계산하는 수식:
```
ω = 2π × N / (Z × ΔT)
```
- ω: 각속도 (rad/s)
- N: 샘플링 주기 동안의 펄스 수
- Z: 회전당 펄스 수 (pulses per revolution)
- ΔT: 샘플링 주기 (sec)

**선속도 계산:**
```
v = ω × r
```
- v: 선속도 (m/s)
- r: 휠 반지름 (m)

In [ ]:
def pulse_timing_method(self, time: np.ndarray, pulses: np.ndarray, pulses_per_revolution: int) ->np.ndarray:

	prev_time = time[0]
	prev_pulse = pulses[0]
	curr_est_ang_vel = 0.0
	est_ang_vel = np.zeros(len(time))

	for idx in range(1,len(time)):
		if pulses[idx] == 1 and prev_pulse == 0: 
			delta_time = time[idx] - prev_time

			if delta_time < 1e-3:
				delta_time = 1e-3

			curr_est_ang_vel = 2 * np.pi / (pulses_per_revolution * delta_time)
			prev_time = time[idx]
	
		est_ang_vel[idx] = curr_est_ang_vel
		prev_pulse = pulses[idx]

	return est_ang_vel

### 4. DR(Dead Reckoning)을 통한 위치 추정

Dead Reckoning(DR): **IMU 센서(자이로의 yaw rate)와 속도 정보(Wheel Odometry를 통해 취득)를 활용하여 차량의 위치를 적분적으로 추정하는 방법**
GPS/INS 데이터로부터 얻은 초기 위치와 속도를 바탕으로, IMU yaw rate와 속도를 적분하여 DR 경로를 계산

**장점**: GPS 없이도 연속적으로 위치 추정 가능

**단점**: 센서 노이즈와 바이어스로 인해 시간이 지남에 따라 누적 오차 발생

### DR 위치 추정 과정
1. **센서 데이터 동기화**  
   - IMU 시간축 기준으로 Wheel Odometry 속도 데이터를 보간(interpolation)  
   - `np.interp()`를 사용하여 imu_time 기준 속도 배열 생성  

2. **초기 상태 설정**  
   - 시작 위도, 경도, 방위각(azimuth) 추출  
   - 지구 반경 (R = 6378137m) 사용하여 위도/경도 변화량 계산 준비  

3. **방향(Heading) 업데이트**  
   - IMU yaw rate $(\omega)$ 적분  
   
   $\theta_k = \theta_{k-1} + \omega \cdot \Delta t$  

4. **변위 계산**  
   - 선속도 ($v$) 와 방향 $(\theta)$ 를 이용한 이동 거리 계산  
   
   $dx = v \cdot \cos(\theta) \cdot \Delta t$  
   
   $dy = v \cdot \sin(\theta) \cdot \Delta t$  

5. **위도/경도 업데이트**  
   - 지구 반경을 고려한 위도, 경도 증분 계산  
   
   $\Delta \phi = \frac{dy}{R}, \quad$
   $\Delta \lambda = \frac{dx}{R \cdot \cos(\phi)}$
     
   - 최종 위치 누적

   $\phi_k = \phi_{k-1} + \Delta \phi, \quad$
   $\lambda_k = \lambda_{k-1} + \Delta \lambda$


In [ ]:
    def calculate_dr_trajectory(self, imu_data: dict, inspvax_data: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        # 데이터 추출 및 시간 정규화
        imu_time = np.array(imu_data['time'])
        imu_yaw_rate = np.array([d.angular_velocity.z for d in imu_data['data']])
        inspvax_time = np.array(inspvax_data['time'])
        north_vel = np.array([d.north_velocity for d in inspvax_data['data']])
        east_vel = np.array([d.east_velocity for d in inspvax_data['data']])
        gt_velocity_raw = np.sqrt(north_vel**2 + east_vel**2)
        
        # 시간 동기화
        start_time = imu_time[0]
        imu_time -= start_time
        inspvax_time -= start_time

        velocity_interp = np.interp(imu_time, inspvax_time, gt_velocity_raw)

        # DR 위치 추정 및 초기 위치 설정 (위도/경도 직접 계산)
        R = 6378137.0  # 지구 반지름 (m)

        # GPS에서 도로 받음 → 라디안으로 변환 → 삼각함수로 계산 → 다시 도로 저장
        lat_rad = np.deg2rad(inspvax_data['data'][0].latitude)
        lon_rad = np.deg2rad(inspvax_data['data'][0].longitude)
        theta = np.deg2rad(90.0 - inspvax_data['data'][0].azimuth)

        # DR 위치 저장 ndarrays
        N = len(imu_time)
        dr_lat=np.zeros(N)
        dr_lon=np.zeros(N)
        dr_lat[0] = np.rad2deg(lat_rad)
        dr_lon[0] = np.rad2deg(lon_rad)

        for i in range(1, len(imu_time)):
            
            dt = imu_time[i] - imu_time[i-1]

            theta += imu_yaw_rate[i] * dt # 방위각 업데이트     
            dx = velocity_interp[i] * np.cos(theta) * dt # x축 이동 거리 업데이트
            dy = velocity_interp[i] * np.sin(theta) * dt # y축 이동 거리 업데이트
                       
            # 위도, 경도 업데이트
            lat_rad += dy / R
            lon_rad += dx / (R * np.cos(lat_rad))
            
            # DR 위치 저장
            dr_lat[i] = np.rad2deg(lat_rad)
            dr_lon[i] = np.rad2deg(lon_rad)
        
        return dr_lat, dr_lon, imu_time